# 01 — NL to IR (Natural Language → Intermediate Representation)

This notebook converts a natural-language test case into a structured IR (Intermediate Representation).

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis

This notebook:
- Loads a TodoMVC natural-language test case
- Converts it into IR using a minimal rule-based parser
- Saves the IR for downstream notebooks


In [14]:
from pathlib import Path
import json
import re


## 1. Load Natural-Language Test Case

We load a simple TodoMVC test case from the sample-data directory.


In [15]:
sample_path = Path("../sample-data/test_cases/todomvc_add_and_complete.txt")

with open(sample_path, "r") as f:
    test_case_text = f.read()

test_case_text


'Add “buy milk”, add “get gas in Susie’s car”, then complete “buy milk”, and complete “get gas in Susie’s car”.\n'

## 2. Minimal NL → IR Converter

This is a lightweight rule-based parser designed to support the TodoMVC demo.

Supported actions:
- Add a todo item
- Press Enter
- Verify the item appears


In [16]:
def generate_ir_from_text(text):
    steps = []

    if "buy milk" in text.lower():
        steps.append({"action": "input", "target": "new_todo_input", "value": "buy milk"})
        steps.append({"action": "press", "target": "new_todo_input", "key": "Enter"})

    if "get gas" in text.lower():
        steps.append({"action": "input", "target": "new_todo_input", "value": "get gas in Susie’s car"})
        steps.append({"action": "press", "target": "new_todo_input", "key": "Enter"})

    if "complete" in text.lower() and "buy milk" in text.lower():
        steps.append({"action": "click", "target": "todo_item_1_checkbox"})

    if "complete" in text.lower() and "susie" in text.lower():
        steps.append({"action": "click", "target": "todo_item_2_checkbox"})

    return {
        "test_name": "todomvc_add_and_complete_items",
        "description": text,
        "steps": steps,
        "metadata": {"priority": "high"}
    }


## 3. Generate IR


In [17]:
ir = generate_ir_from_text(test_case_text)
ir


{'test_name': 'todomvc_add_and_complete_items',
 'description': 'Add “buy milk”, add “get gas in Susie’s car”, then complete “buy milk”, and complete “get gas in Susie’s car”.\n',
 'steps': [{'action': 'input',
   'target': 'new_todo_input',
   'value': 'buy milk'},
  {'action': 'press', 'target': 'new_todo_input', 'key': 'Enter'},
  {'action': 'input',
   'target': 'new_todo_input',
   'value': 'get gas in Susie’s car'},
  {'action': 'press', 'target': 'new_todo_input', 'key': 'Enter'},
  {'action': 'click', 'target': 'todo_item_1_checkbox'},
  {'action': 'click', 'target': 'todo_item_2_checkbox'}],
 'metadata': {'priority': 'high'}}

## 4. Save IR for Downstream Notebooks

The IR is saved to `../sample-data/ir_examples/todomvc_ir.json`.


In [18]:
output_path = Path("../sample-data/ir_examples/todomvc_ir.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(ir, f, indent=4)

output_path


PosixPath('../sample-data/ir_examples/todomvc_ir.json')